Basics of lancedb

In [1]:
import lancedb

db = lancedb.connect("knowledge_base")
db

LanceDBConnection(uri='/Users/juliusfaltskoglund/dev/github/AI_OPS_julius_falt_skoglund_DE25/code-alongs/09_lancedb/knowledge_base')

In [2]:
db.uri

'/Users/juliusfaltskoglund/dev/github/AI_OPS_julius_falt_skoglund_DE25/code-alongs/09_lancedb/knowledge_base'

## Create table

In [3]:
import json
with open("data/animals_text_embeddings.json") as f:
    data = json.load(f)

# This data has vector embedding already created
# For this it is generated by AI, but in realitu if you use other vector databases that 
# don't automatically create embeddings you need to create the embeddings yourself
data

[{'text': 'A small brown dog running.', 'vector': [0.12, 0.85, 0.33]},
 {'text': 'A cat resting quietly on a sofa.', 'vector': [0.4, 0.91, 0.1]},
 {'text': 'A large gray elephant drinking water.',
  'vector': [0.88, 0.22, 0.55]},
 {'text': 'A fast cheetah sprinting across the savannah.',
  'vector': [0.95, 0.12, 0.72]},
 {'text': 'A colorful parrot perched on a branch.',
  'vector': [0.25, 0.66, 0.81]},
 {'text': 'A frog sitting on a lily pad.', 'vector': [0.14, 0.44, 0.27]}]

In [4]:
data[0].get("text")

'A small brown dog running.'

In [5]:
data[0].get("vector")

[0.12, 0.85, 0.33]

In [6]:
db.create_table("animals_text", exist_ok = True, data=data)
db["animals_text"]

LanceTable(name='animals_text', _conn=LanceDBConnection(uri='/Users/juliusfaltskoglund/dev/github/AI_OPS_julius_falt_skoglund_DE25/code-alongs/09_lancedb/knowledge_base'))

In [7]:
db["animals_text"].head()

pyarrow.Table
text: string
vector: fixed_size_list<item: float>[3]
  child 0, item: float
----
text: [["A small brown dog running.","A cat resting quietly on a sofa.","A large gray elephant drinking water.","A fast cheetah sprinting across the savannah.","A colorful parrot perched on a branch."]]
vector: [[[0.12,0.85,0.33],[0.4,0.91,0.1],[0.88,0.22,0.55],[0.95,0.12,0.72],[0.25,0.66,0.81]]]

In [8]:
db["animals_text"].to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"


In [9]:
more_data = [
    {"text": "A panda eating bamboo peacefully.", "vector": [0.51, 0.37, 0.82]},
    {"text": "A lion roaring loudly on a rock.", "vector": [0.93, 0.18, 0.41]},
]
db["animals_text"].add(more_data)

AddResult(version=2)

In [10]:
db["animals_text"].to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


## Drop a table

In [11]:
db

LanceDBConnection(uri='/Users/juliusfaltskoglund/dev/github/AI_OPS_julius_falt_skoglund_DE25/code-alongs/09_lancedb/knowledge_base')

In [12]:
from lancedb.pydantic import LanceModel

class CoolSchema(LanceModel):
    cool_name: str 
    cool_number: int = 42

# mode overwrite -> replace in SQL 
# exist_ok = True -> if not exists in SQL
db.create_table("cool_table", schema=CoolSchema, mode="overwrite")

[2026-11-12T12:17:51Z WARN  lance::dataset::write::insert] No existing dataset at /Users/juliusfaltskoglund/dev/github/AI_OPS_julius_falt_skoglund_DE25/code-alongs/09_lancedb/knowledge_base/cool_table.lance, it will be created


LanceTable(name='cool_table', _conn=LanceDBConnection(uri='/Users/juliusfaltskoglund/dev/github/AI_OPS_julius_falt_skoglund_DE25/code-alongs/09_lancedb/knowledge_base'))

In [13]:
import pandas as pd 
cool_data = [{"cool_name": "pi", "cool_number": 3.141592}, {"cool_name": "e", "cool_number": 2.713}]
cool_df = pd.DataFrame(cool_data)
cool_df

,cool_name,cool_number
0,pi,3.141592
1,e,2.713000


In [14]:
db["cool_table"].add(cool_df)

AddResult(version=2)

In [15]:
db["cool_table"].to_pandas()

,cool_name,cool_number
0,pi,3
1,e,2


In [16]:
# Common flow in many vector databases
# natural language _> embedder -> query_vector
query_vector = [0.5, 0.2, 0.9]
db["animals_text"].search(query=query_vector).limit(3).to_pandas()

,text,vector,_distance
0,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]",0.0354
1,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]",0.2413
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]",0.2673


## Embeddings API

In [17]:
from lancedb.pydantic import Vector
from lancedb.embeddings import get_registry
from dotenv import load_dotenv

load_dotenv()

model = get_registry().get("cohere").create(name="embed-multilingual-v3.0")
class JokeModel(LanceModel):
    jokes: str = model.SourceField()
    vector: Vector(model.ndims()) = model.VectorField()

print(f"{model.ndims()}")

db.create_table("jokes", schema=JokeModel, exist_ok = True)

1024


LanceTable(name='jokes', _conn=LanceDBConnection(uri='/Users/juliusfaltskoglund/dev/github/AI_OPS_julius_falt_skoglund_DE25/code-alongs/09_lancedb/knowledge_base'))

In [18]:
df=pd.read_json("data/jokes.json")
df.head()

,jokes
0,Parallel lines have so much in common—it’s sad...
1,"ETL stands for “Extract, Transform, Leave for ..."
2,What do you call a snake that runs your script...
3,"Gold walks into a bar. The bartender says, “Au..."
4,C# devs don’t argue; they just throw exceptions.


In [19]:
db["jokes"].add(df)

AddResult(version=2)

In [20]:
db["jokes"].to_pandas().head()

,jokes,vector
0,Parallel lines have so much in common—it’s sad...,"[-0.00301, 0.001863, -0.014427, 0.051575, -0.0..."
1,"ETL stands for “Extract, Transform, Leave for ...","[0.01239, 0.0047, -0.091431, 0.080627, -0.0505..."
2,What do you call a snake that runs your script...,"[0.019012, 0.044525, -0.021454, 0.02887, -0.00..."
3,"Gold walks into a bar. The bartender says, “Au...","[0.019958, 0.060089, 0.012733, 0.024048, -0.07..."
4,C# devs don’t argue; they just throw exceptions.,"[0.013275, 0.002462, -0.057007, 0.060883, -0.0..."


In [21]:
db["jokes"].search("Ge mig kemi skämt").limit(4).to_pandas()

,jokes,vector,_distance
0,I told a chemistry joke… there was no reaction.,"[-0.032745, 0.018295, 0.039368, 0.001193, -0.0...",0.843234
1,Oxygen and Magnesium got together—OMg!,"[-0.041138, -0.004559, 0.007397, 0.006527, -0....",1.041996
2,Chemists have all the solutions… mostly in bea...,"[0.00139, 0.030945, -0.001131, 0.021011, -0.05...",1.072914
3,Why did the chemist ground his kids? Because t...,"[-0.00824, 0.031525, -0.019638, 0.015228, -0.0...",1.085388


In [22]:
db["jokes"].search("Ge mig matematik skämt").limit(4).to_pandas()

,jokes,vector,_distance
0,My math teacher said I’m average… how mean!,"[-0.002581, 0.031525, 0.013107, 0.030991, -0.0...",0.942483
1,Why’s 6 afraid of 7? Because 7 8 9.,"[0.005367, 0.020432, -0.002979, 0.016785, -0.0...",0.980481
2,I told a chemistry joke… there was no reaction.,"[-0.032745, 0.018295, 0.039368, 0.001193, -0.0...",1.092561
3,"Dear algebra, please stop asking us to find yo...","[0.011345, 0.004944, -0.019363, 0.060486, -0.0...",1.133865


## Hybrid search

In [23]:
from lancedb.index import FTS
# Keyword based matching on jokes column
db["jokes"].create_index("jokes", config=FTS(), replace=True)

In [24]:
db["jokes"].search("Give me a math jokes", 
    query_type="hybrid", 
    vector_column_name="vector", 
    fts_columns="jokes",
).limit(5).to_pandas()

,jokes,vector,_relevance_score
0,My math teacher said I’m average… how mean!,"[-0.002581, 0.031525, 0.013107, 0.030991, -0.0...",0.032258
1,Why’s 6 afraid of 7? Because 7 8 9.,"[0.005367, 0.020432, -0.002979, 0.016785, -0.0...",0.016393
2,I told a chemistry joke… there was no reaction.,"[-0.032745, 0.018295, 0.039368, 0.001193, -0.0...",0.016393
3,The C# compiler walked into a bar. The bartend...,"[0.003672, 0.006752, -0.076904, 0.026688, -0.0...",0.015873
4,"Dear algebra, please stop asking us to find yo...","[0.011345, 0.004944, -0.019363, 0.060486, -0.0...",0.015625


In [25]:
from lancedb import rerankers

reranker = rerankers.RRFReranker()

db["jokes"].search("Give me a math jokes", 
    query_type="hybrid", 
    vector_column_name="vector", 
    fts_columns="jokes",
).rerank(reranker).limit(5).to_pandas()

,jokes,vector,_relevance_score
0,My math teacher said I’m average… how mean!,"[-0.002581, 0.031525, 0.013107, 0.030991, -0.0...",0.032258
1,Why’s 6 afraid of 7? Because 7 8 9.,"[0.005367, 0.020432, -0.002979, 0.016785, -0.0...",0.016393
2,I told a chemistry joke… there was no reaction.,"[-0.032745, 0.018295, 0.039368, 0.001193, -0.0...",0.016393
3,The C# compiler walked into a bar. The bartend...,"[0.003672, 0.006752, -0.076904, 0.026688, -0.0...",0.015873
4,"Dear algebra, please stop asking us to find yo...","[0.011345, 0.004944, -0.019363, 0.060486, -0.0...",0.015625
